# Category-aware recommendations

The engine selects one result per eligible category slot. The synthetic fixture has synthetic catalog embeddings, so its scores serve only as a workflow demonstration.


In [ ]:
import os, sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ARCHITECTURE.md').is_file())
sys.path.insert(0, str(root))
from pipeline.prepare_dataset import read_json
fixture = root / 'data/processed/fixture'
manifest_path = Path(os.environ.get('STYLIST_MANIFEST', fixture / 'manifest.json'))
manifest = read_json(manifest_path)
image_root = Path(os.environ.get('STYLIST_IMAGE_ROOT', fixture))
print('Dataset:', manifest['dataset'], '| Revision:', manifest['revision'])

In [ ]:
from backend.app.recommendations.engine import Catalog, DEFAULT_POLICY
snapshot_env = os.environ.get('STYLIST_SNAPSHOT')
snapshot = Path(snapshot_env if snapshot_env else read_json(fixture / 'fixture.json')['snapshot'])
catalog = Catalog(snapshot)

import matplotlib.pyplot as plt
from PIL import Image
from backend.app.ml.encoder import ClipEncoder
item = catalog.items[0]
os.environ.setdefault('HF_HOME', str(root / 'data/raw/hf-cache'))
with Image.open(image_root / item['image']) as image:
    query = ClipEncoder()([image.convert('RGB')])[0]
matches, warnings = catalog.recommend(query, item['category'], query_id=item['id'])
fig, axes = plt.subplots(1, len(matches) + 1, squeeze=False)
for ax, row, label in zip(axes.flat, [item] + [catalog.items[catalog.rows[m.item_id]] for m in matches],
                          ['Query'] + [f'{m.category} {m.similarity:.3f}' for m in matches]):
    with Image.open(image_root / row['image']) as image:
        ax.imshow(image)
    ax.set_title(label)
    ax.axis('off')
plt.show()
print(warnings)